# Modelos de Boosting

Se comparan AdaBoost, Gradient Boosting, XGBoost, LightGBM y CatBoost. La búsqueda de hiperparámetros es moderada y se realiza solo sobre entrenamiento.


## Librerías


In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor


## Datos procesados


In [ ]:
bank_train_con_duration = pd.read_csv("data/processed/serie_computacional/bank_train_con_duration.csv")
bank_test_con_duration = pd.read_csv("data/processed/serie_computacional/bank_test_con_duration.csv")
bank_train_sin_duration = pd.read_csv("data/processed/serie_computacional/bank_train_sin_duration.csv")
bank_test_sin_duration = pd.read_csv("data/processed/serie_computacional/bank_test_sin_duration.csv")

consumo_train = pd.read_csv("data/processed/serie_computacional/power_train_hourly.csv")
consumo_test = pd.read_csv("data/processed/serie_computacional/power_test_hourly.csv")

resumen_datos = pd.DataFrame([
    {"Dataset": "Bank Marketing", "Escenario": "con duration", "Entrenamiento": len(bank_train_con_duration), "Prueba": len(bank_test_con_duration), "Variables": bank_train_con_duration.shape[1] - 1},
    {"Dataset": "Bank Marketing", "Escenario": "sin duration", "Entrenamiento": len(bank_train_sin_duration), "Prueba": len(bank_test_sin_duration), "Variables": bank_train_sin_duration.shape[1] - 1},
    {"Dataset": "Electric Power", "Escenario": "regresión horaria", "Entrenamiento": len(consumo_train), "Prueba": len(consumo_test), "Variables": consumo_train.shape[1] - 2},
])

resumen_datos


## Funciones auxiliares


In [ ]:
def preparar_preprocesador(X):
    variables_numericas = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col])]
    variables_categoricas = [col for col in X.columns if col not in variables_numericas]

    try:
        codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        codificador = OneHotEncoder(handle_unknown="ignore", sparse=False)

    pasos = []
    if variables_numericas:
        pasos.append(("numericas", SimpleImputer(strategy="median"), variables_numericas))
    if variables_categoricas:
        pasos.append(("categoricas", codificador, variables_categoricas))

    return ColumnTransformer(pasos), variables_numericas, variables_categoricas


def separar_bank(entrenamiento, prueba):
    X_entrenamiento = entrenamiento.drop(columns=["y"])
    y_entrenamiento = entrenamiento["y"]
    X_prueba = prueba.drop(columns=["y"])
    y_prueba = prueba["y"]
    return X_entrenamiento, y_entrenamiento, X_prueba, y_prueba


def separar_consumo(entrenamiento, prueba):
    objetivo = "Global_active_power"
    columnas_excluidas = [objetivo, "datetime"]
    X_entrenamiento = entrenamiento.drop(columns=columnas_excluidas)
    y_entrenamiento = entrenamiento[objetivo]
    X_prueba = prueba.drop(columns=columnas_excluidas)
    y_prueba = prueba[objetivo]
    return X_entrenamiento, y_entrenamiento, X_prueba, y_prueba


def evaluar_clasificacion(modelo, nombre, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba, escenario):
    preprocesador, numericas, categoricas = preparar_preprocesador(X_entrenamiento)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    inicio = time.perf_counter()
    flujo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ajuste = time.perf_counter() - inicio
    predicciones = flujo.predict(X_prueba)
    probabilidades = flujo.predict_proba(X_prueba)[:, 1] if hasattr(flujo, "predict_proba") else None

    return flujo, {
        "Dataset": "Bank Marketing",
        "Escenario": escenario,
        "Modelo": nombre,
        "F1 de la clase positiva": f1_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "Precisión": precision_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "Sensibilidad": recall_score(y_prueba, predicciones, pos_label=1, zero_division=0),
        "AUC-ROC": roc_auc_score(y_prueba, probabilidades) if probabilidades is not None else np.nan,
        "Exactitud": accuracy_score(y_prueba, predicciones),
        "Tiempo de ajuste (s)": tiempo_ajuste,
        "Variables originales": X_entrenamiento.shape[1],
        "Variables numéricas": len(numericas),
        "Variables categóricas": len(categoricas),
    }


def evaluar_regresion(modelo, nombre, X_entrenamiento, y_entrenamiento, X_prueba, y_prueba):
    preprocesador, numericas, categoricas = preparar_preprocesador(X_entrenamiento)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    inicio = time.perf_counter()
    flujo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ajuste = time.perf_counter() - inicio
    predicciones = flujo.predict(X_prueba)
    mse = mean_squared_error(y_prueba, predicciones)

    return flujo, {
        "Dataset": "Individual Household Electric Power Consumption",
        "Modelo": nombre,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_prueba, predicciones),
        "MSE": mse,
        "R²": r2_score(y_prueba, predicciones),
        "Tiempo de ajuste (s)": tiempo_ajuste,
        "Variables originales": X_entrenamiento.shape[1],
        "Variables numéricas": len(numericas),
        "Variables categóricas": len(categoricas),
    }


## Búsqueda moderada de hiperparámetros

En clasificación se usa validación cruzada estratificada y F1 de la clase positiva. En regresión se utiliza validación temporal con RMSE. El test queda reservado para la evaluación final.


In [ ]:
validacion_clasificacion = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
validacion_regresion = TimeSeriesSplit(n_splits=3)


def modelos_boosting_clasificacion():
    return {
        "AdaBoost": AdaBoostClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "XGBoost": XGBClassifier(objective="binary:logistic", eval_metric="logloss", random_state=42, n_jobs=1, verbosity=0),
        "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
        "CatBoost": CatBoostClassifier(random_seed=42, verbose=False, allow_writing_files=False),
    }


def modelos_boosting_regresion():
    return {
        "AdaBoost": AdaBoostRegressor(random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
        "XGBoost": XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=1, verbosity=0),
        "LightGBM": LGBMRegressor(random_state=42, verbose=-1),
        "CatBoost": CatBoostRegressor(random_seed=42, verbose=False, allow_writing_files=False),
    }


espacios_busqueda = {
    "AdaBoost": {
        "modelo__n_estimators": [50, 100, 200],
        "modelo__learning_rate": [0.03, 0.05, 0.1, 0.5],
    },
    "Gradient Boosting": {
        "modelo__n_estimators": [50, 100, 200],
        "modelo__learning_rate": [0.03, 0.05, 0.1],
        "modelo__max_depth": [2, 3],
    },
    "XGBoost": {
        "modelo__n_estimators": [100, 200, 300],
        "modelo__learning_rate": [0.03, 0.05, 0.1],
        "modelo__max_depth": [2, 3, 5],
    },
    "LightGBM": {
        "modelo__n_estimators": [100, 200, 300],
        "modelo__learning_rate": [0.03, 0.05, 0.1],
        "modelo__num_leaves": [7, 15, 31],
    },
    "CatBoost": {
        "modelo__iterations": [100, 200, 300],
        "modelo__learning_rate": [0.03, 0.05, 0.1],
        "modelo__depth": [3, 5, 7],
    },
}


def buscar_modelo(modelo, parametros, X, y, validacion, metrica):
    preprocesador, _, _ = preparar_preprocesador(X)
    flujo = Pipeline([("preprocesamiento", preprocesador), ("modelo", modelo)])
    busqueda = RandomizedSearchCV(
        flujo,
        param_distributions=parametros,
        n_iter=8,
        scoring=metrica,
        cv=validacion,
        random_state=42,
        n_jobs=1,
        refit=True,
    )
    inicio = time.perf_counter()
    busqueda.fit(X, y)
    tiempo_busqueda = time.perf_counter() - inicio
    return busqueda, tiempo_busqueda


## Clasificación


In [ ]:
filas_busqueda = []
filas_test = []

for escenario, entrenamiento, prueba in [
    ("con duration", bank_train_con_duration, bank_test_con_duration),
    ("sin duration", bank_train_sin_duration, bank_test_sin_duration),
]:
    X_entrenamiento, y_entrenamiento, X_prueba, y_prueba = separar_bank(entrenamiento, prueba)
    for nombre, modelo in modelos_boosting_clasificacion().items():
        busqueda, tiempo_busqueda = buscar_modelo(
            modelo,
            espacios_busqueda[nombre],
            X_entrenamiento,
            y_entrenamiento,
            validacion_clasificacion,
            "f1",
        )
        predicciones = busqueda.predict(X_prueba)
        probabilidades = busqueda.predict_proba(X_prueba)[:, 1]
        filas_busqueda.append({
            "Tarea": "clasificación",
            "Escenario": escenario,
            "Modelo": nombre,
            "Configuraciones probadas": 8,
            "Estrategia de validación": "StratifiedKFold(n_splits=3)",
            "Mejores parámetros": str(busqueda.best_params_),
            "Media de validación": busqueda.best_score_,
            "Tiempo de búsqueda (s)": tiempo_busqueda,
        })
        filas_test.append({
            "Dataset": "Bank Marketing",
            "Escenario": escenario,
            "Modelo": nombre,
            "F1 de la clase positiva": f1_score(y_prueba, predicciones, pos_label=1, zero_division=0),
            "Precisión": precision_score(y_prueba, predicciones, pos_label=1, zero_division=0),
            "Sensibilidad": recall_score(y_prueba, predicciones, pos_label=1, zero_division=0),
            "AUC-ROC": roc_auc_score(y_prueba, probabilidades),
            "Exactitud": accuracy_score(y_prueba, predicciones),
            "Tiempo de búsqueda (s)": tiempo_busqueda,
            "Mejores parámetros": str(busqueda.best_params_),
        })

busqueda_boosting_clasificacion = pd.DataFrame(filas_busqueda)
resultados_boosting_clasificacion = pd.DataFrame(filas_test)

display(busqueda_boosting_clasificacion)
resultados_boosting_clasificacion


## Regresión


In [ ]:
filas_busqueda = []
filas_test = []
X_entrenamiento, y_entrenamiento, X_prueba, y_prueba = separar_consumo(consumo_train, consumo_test)

for nombre, modelo in modelos_boosting_regresion().items():
    busqueda, tiempo_busqueda = buscar_modelo(
        modelo,
        espacios_busqueda[nombre],
        X_entrenamiento,
        y_entrenamiento,
        validacion_regresion,
        "neg_root_mean_squared_error",
    )
    predicciones = busqueda.predict(X_prueba)
    mse = mean_squared_error(y_prueba, predicciones)
    filas_busqueda.append({
        "Tarea": "regresión",
        "Escenario": "consumo eléctrico horario",
        "Modelo": nombre,
        "Configuraciones probadas": 8,
        "Estrategia de validación": "TimeSeriesSplit(n_splits=3)",
        "Mejores parámetros": str(busqueda.best_params_),
        "Media de validación": -busqueda.best_score_,
        "Tiempo de búsqueda (s)": tiempo_busqueda,
    })
    filas_test.append({
        "Dataset": "Individual Household Electric Power Consumption",
        "Modelo": nombre,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_prueba, predicciones),
        "MSE": mse,
        "R²": r2_score(y_prueba, predicciones),
        "Tiempo de búsqueda (s)": tiempo_busqueda,
        "Mejores parámetros": str(busqueda.best_params_),
    })

busqueda_boosting_regresion = pd.DataFrame(filas_busqueda)
resultados_boosting_regresion = pd.DataFrame(filas_test)

display(busqueda_boosting_regresion)
resultados_boosting_regresion


## Guardado y figuras


In [ ]:
Path("reports/tables/serie_computacional").mkdir(parents=True, exist_ok=True)
Path("reports/figures/serie_computacional").mkdir(parents=True, exist_ok=True)

busqueda_boosting_clasificacion.to_csv("reports/tables/serie_computacional/boosting_busqueda_clasificacion.csv", index=False)
busqueda_boosting_regresion.to_csv("reports/tables/serie_computacional/boosting_busqueda_regresion.csv", index=False)
resultados_boosting_clasificacion.to_csv("reports/tables/serie_computacional/boosting_clasificacion_metricas.csv", index=False)
resultados_boosting_regresion.to_csv("reports/tables/serie_computacional/boosting_regresion_metricas.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4))
grafico = resultados_boosting_clasificacion.copy()
grafico["Etiqueta"] = grafico["Modelo"] + "\n" + grafico["Escenario"]
ax.bar(grafico["Etiqueta"], grafico["F1 de la clase positiva"], color="#4c78a8")
ax.set_title("Modelos de Boosting: F1 de la clase positiva")
ax.set_ylabel("F1 de la clase positiva")
ax.tick_params(axis="x", rotation=60)
fig.tight_layout()
fig.savefig("reports/figures/serie_computacional/boosting_clasificacion_f1.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(resultados_boosting_regresion["Modelo"], resultados_boosting_regresion["RMSE"], color="#f58518")
ax.set_title("Modelos de Boosting: RMSE")
ax.set_ylabel("RMSE")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig("reports/figures/serie_computacional/boosting_regresion_rmse.png", dpi=150)
plt.show()

pd.DataFrame([
    {"archivo": "boosting_busqueda_clasificacion.csv"},
    {"archivo": "boosting_busqueda_regresion.csv"},
    {"archivo": "boosting_clasificacion_metricas.csv"},
    {"archivo": "boosting_regresion_metricas.csv"},
])
